In [2]:
import sqlite3
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import statsmodels.formula.api as smf
from pathlib import Path

import sys; sys.path.insert(0, '..')
from src.palette import register, content_colors, colorway, CONTENT_ORDER

CONTENT_COLORS = register('light')

In [3]:
conn = sqlite3.connect('../data/lafc_content.db')

In [4]:
with open('../sql/videos_vs_lafc_match_context.sql') as f:
    query = f.read()

df = pd.read_sql(query, conn)
df['content_type'] = df['content_type'].fillna('no_playlist')
df['playlist'] = df['playlist'].fillna('(no playlist)')

In [5]:
df.head()

,video_id,title,description,published_at,duration,view_count,like_count,comment_count,engagement_rate,format,...,goals_for,goals_against,lafc_points,lafc_played,lafc_wins,opp_points,opp_played,opp_wins,days_since_match,days_until_match
0,IxrFLgowFd4,Armindo Sieb is Black & Gold.,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T16:40:02Z,PT52S,275,23,7,0.10909,horizontal,...,1.0,1.0,33.0,18.0,10.0,33.0,16.0,10.0,11.72,NaN
1,HugEGKBw0kk,LAFC vs QRO | Postmatch Media,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T09:02:51Z,PT14M19S,306,22,26,0.15686,horizontal,...,1.0,1.0,33.0,18.0,10.0,33.0,16.0,10.0,11.40,NaN
2,pLVoxNTyGLI,The top scorer in Leagues Cup history 📈,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T07:30:24Z,PT15S,6287,169,12,0.02879,short,...,1.0,1.0,33.0,18.0,10.0,33.0,16.0,10.0,11.33,NaN
3,wz6UrdGQWjY,BOUANGA EQUALIZER 💥,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T07:15:01Z,PT13S,3940,91,4,0.02411,short,...,1.0,1.0,33.0,18.0,10.0,33.0,16.0,10.0,11.32,NaN
4,SqgJkPzCN6Y,Denis Bouanga equalizes against Querétaro,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T07:02:06Z,PT13S,1712,55,6,0.03563,horizontal,...,1.0,1.0,33.0,18.0,10.0,33.0,16.0,10.0,11.31,NaN


In [6]:
published_at = pd.to_datetime(df['published_at'], format='ISO8601', utc=True)
df['quarter'] = published_at.dt.to_period('Q')

display(df[['title', 'quarter','published_at']].sort_values('quarter', ascending=False))

/var/folders/g4/h7hn6rpd2hv1lz_lqhhb8qzw0000gn/T/ipykernel_36485/531845570.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['quarter'] = published_at.dt.to_period('Q')


,title,quarter,published_at
0,Armindo Sieb is Black & Gold.,2026Q3,2026-08-13T16:40:02Z
70,엘 트라피코 골 ✅ 1호골 ✅ 손흥민 골 ✅,2026Q3,2026-07-22T23:32:24Z
81,A Night To Remember | LAG vs LAFC,2026Q3,2026-07-18T22:58:28Z
80,LAFC Weekly | Episode 15 | 2026,2026Q3,2026-07-19T01:00:21Z
79,Son Heung-Min | EVERY ANGLE of his derby goal ...,2026Q3,2026-07-20T07:17:45Z
...,...,...,...
3638,LAFC Colors & Crest Launch,2016Q1,2016-01-14T19:59:03Z
3637,LAFC Owner Ruben Gnanalingam on Bloomberg TV,2016Q1,2016-02-29T20:26:26Z
3645,John Thorrington announcement on SportsCenter,2015Q4,2015-12-09T23:54:13Z
3646,Building Together: LAFC Stadium Workshop,2015Q4,2015-11-24T19:50:33Z


In [7]:
df.groupby(['quarter', 'format']).size().unstack(fill_value=0)

format,horizontal,live,short
quarter,,,
2015Q4,3,0,0
2016Q1,9,0,0
2016Q2,2,0,0
2016Q3,6,0,0
2016Q4,6,0,0
2017Q1,7,0,0
2017Q2,1,1,0
2017Q3,15,0,0
2017Q4,16,0,0


In [8]:
quarterly_df = df.groupby(['quarter', 'format']).size().unstack(fill_value=0)
FORMATS = list(quarterly_df.columns)
quarterly_df['total'] = quarterly_df[FORMATS].sum(axis=1)
quarterly_df['short_pct_of_total'] = ((quarterly_df['short'] / quarterly_df['total']) * 100).round(2)
quarterly_df

format,horizontal,live,short,total,short_pct_of_total
quarter,,,,,
2015Q4,3,0,0,3,0.00
2016Q1,9,0,0,9,0.00
2016Q2,2,0,0,2,0.00
2016Q3,6,0,0,6,0.00
2016Q4,6,0,0,6,0.00
2017Q1,7,0,0,7,0.00
2017Q2,1,1,0,2,0.00
2017Q3,15,0,0,15,0.00
2017Q4,16,0,0,16,0.00


In [ ]:
quarterly_uploads_fig = px.bar(
    quarterly_df, x=quarterly_df.index.to_timestamp(), y=['short', 'horizontal', 'live'],
    labels={'value': 'Uploads', 'variable': 'Format', 'x': ''},
    title='Number and Format of Video Uploads by Quarter'
)

quarterly_uploads_fig.show()



In [ ]:
#quarterly_uploads_fig.write_image('../docs/img/uploads_by_format_quarterly.png',
                width=1000, height=500, scale=2)